# Part 6 - The Four Memories: typed stores the agent edits itself

*Agents from First Principles, Part 6.*

Part 5 gave the agent an episodic buffer: a flat list of verbal lessons. That was enough to carry one kind of knowledge across trials, and it exposes the next gap. A flat buffer cannot tell three different KINDS of knowledge apart:

- what HAPPENED (the user opened a return at 14:02) -- an event
- what is TRUE (the user is Dana, she prefers phone) -- a durable fact
- how to DO a task (returns after the window take a 10% fee) -- a procedure

Stuff all three into one list and retrieval gets noisy, facts get buried under events, and a learned procedure reads like a one-off note. Worse, every tool the agent had through Part 5 was READ-ONLY (RAG Part 19's tools only fetched); the agent could never deliberately UPDATE its own state.

This part fixes both, with two ideas.

**1. FOUR TYPED MEMORIES**, by the kind of knowledge they hold (a cognitive-science taxonomy, used in MemGPT/Letta and others):

- **WORKING** what we are doing right now (volatile scratchpad)
- **SEMANTIC** what is TRUE (durable facts: the user, the world)
- **EPISODIC** what HAPPENED (an event log; this FORMALIZES Part 5's reflection buffer)
- **PROCEDURAL** how to DO a task (learned rules; e.g. Part 5's promoted reflection)

A write-ROUTER classifies each incoming item and sends it to the right store, and each store has its own read path.

**2. MEMORY AS AN ACTION.** `memory_append` and `memory_replace` become first-class TOOLS, declared with the Part 1 contract (typed schema, validated before firing), over labeled CORE blocks (`user_profile`, `task_state`). The controller calls them in-loop to rewrite its own persistent memory, exactly as MemGPT/Letta let an agent edit its core memory. The agent is no longer a read-only consumer of state.

And a placement sketch borrowed from operating systems, **orthogonal** to the taxonomy:

- **CORE** always in context -> the labeled blocks (`user_profile`, `task_state`)
- **RECALL** recent, paged in -> the latest episodic events
- **ARCHIVAL** large, searched -> a big store read via a black-box vector search

The ARCHIVAL read REUSES RAG retrieval AS A BLACK-BOX TOOL. We do NOT re-derive embeddings, similarity, or vector databases here (that was RAG Parts 2-4); vector search is demoted to one read tool the memory system calls.

**Continuity**: same refund world and the Part 5 numbers (`ORD-3300`, a `$200` order returned after the 30-day window, a 10% restocking fee -> `$180`).

> **Runs with no network, no API key, and no third-party dependency.** The deterministic rule router/controller is the offline default; `generate()` is the real-LLM path one env flag away. Set `OPENAI_API_KEY` to see the real banner; the code always falls through to the deterministic rules so output stays reproducible.

Companion script: `four_memories.py`

## Setup

Two standard-library imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the write router's rule matching, the profile normalizer, and the black-box archival tokenizer. No third-party package is needed for the default path, so every cell runs fully offline, exactly as in Parts 1-5.

In [ ]:
import os
import re

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 - The four typed stores

Each store holds one KIND of knowledge. The two CORE blocks (`user_profile`, `task_state`) are the always-in-context labeled memory the agent will edit with tools: `user_profile` is SEMANTIC (durable facts about the user) and `task_state` is WORKING (what we are doing right now). `EPISODIC` and `PROCEDURAL` are append-only logs: one records what HAPPENED, the other records how to DO a task. `ARCHIVAL` is a large knowledge base read ONLY through the black-box vector search in the next step; it never sits in context whole.

In [ ]:
CORE = {
    "user_profile": "(empty)",        # SEMANTIC: durable facts about the user
    "task_state": "(empty)",          # WORKING: what we are doing right now
}
EPISODIC = []                          # what HAPPENED: an event log
PROCEDURAL = []                        # how to DO things: learned rules

# ARCHIVAL: a large knowledge base, read ONLY via the black-box vector_search tool.
ARCHIVAL = [
    "Refunds are accepted within 30 days of purchase if the item is unused.",
    "Returns after the 30-day window are still refundable, minus a 10% restocking fee.",
    "Error E-4042 means the payment was declined by the bank.",
    "Standard shipping takes 3 to 5 business days.",
]

print(f"CORE blocks: {list(CORE)}. ARCHIVAL chunks: {len(ARCHIVAL)}.")

## Step 1 - The archival reader is a BLACK-BOX vector search (reused RAG, not re-derived)

This is the load-bearing scope decision of the part. The archival store is large, so it is read by SEARCH, not by holding it in context. That search is exactly the dense retrieval RAG Parts 2-4 built from scratch, embeddings, cosine similarity, a vector index, and the whole point here is that we do NOT rebuild any of it.

`vector_search` below is a deterministic lexical stand-in that returns the best-matching chunk and a score. Treat it as **opaque**: it stands in for a RAG retriever, and its internals are NOT this part's concern. Memory's job is to know WHEN to page archival in and WHAT to do with the result, not how the similarity is computed. Vector search is demoted from a whole series to one read tool the memory system calls.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")
_STOP = {"the", "a", "an", "of", "to", "for", "on", "in", "is", "are", "what", "how",
         "do", "does", "and", "after", "within"}


def _toks(text):
    return {t[:-1] if len(t) > 3 and t.endswith("s") else t
            for t in _TOKEN_RE.findall(text.lower()) if t not in _STOP}


def vector_search(query):
    """ARCHIVAL read tool: returns the best-matching chunk and a score. This is a
    BLACK BOX standing in for RAG (Parts 2-4); the internals (embeddings,
    similarity, the index) are deliberately NOT re-derived here."""
    q = _toks(query)
    best, best_score = "", 0.0
    for chunk in ARCHIVAL:
        c = _toks(chunk)
        score = len(q & c) / ((len(q) * len(c)) ** 0.5) if q and c else 0.0
        if score > best_score:
            best, best_score = chunk, score
    return best, round(best_score, 2)


print("vector_search ready (opaque archival reader; stands in for RAG).")

## Step 2 - The write ROUTER: classify each item by KIND, send it to the right store

A flat buffer fails because everything lands in one list. The fix is a router that reads an incoming item and decides WHICH of the four kinds it is, then routes it to the matching store. `write_router` is a transparent rule policy you can read, keyed on the same cheap signals an LLM classifier would weigh: first-person self-description goes to SEMANTIC; a fee/policy/rule phrasing goes to PROCEDURAL; a past-tense event goes to EPISODIC; anything else is the current WORKING focus. A real system swaps this body for one `generate()` classification call; the routing contract is identical either way.

`_normalize_profile` is what makes a CORE block editable later: it extracts the durable facts (name, contact preference) into a compact `name: X; prefers Y` phrase, so a later `memory_replace` can target the clean phrase `prefers email` rather than a whole raw sentence. `route_and_store` ties them together: it routes, then writes the item into the chosen store, overwriting a CORE block or appending to a log.

In [ ]:
def write_router(text):
    t = text.lower()
    if re.search(r"\b(i'?m|my name|prefer|i like|call me)\b", t):
        return "semantic", "a durable fact about the user"
    if re.search(r"\b(fee|policy|always|never|rule|step 1|procedure|multiply)\b", t):
        return "procedural", "a reusable how-to rule"
    if re.search(r"\b(opened|asked|clicked|requested|happened|at \d|yesterday|earlier)\b", t):
        return "episodic", "an event that happened"
    return "working", "the current task focus"


def _normalize_profile(text):
    """Extract durable facts into a compact, editable profile (so a later
    memory_replace can target a clean phrase, not a whole raw sentence)."""
    name = re.search(r"i'?m (\w+)", text, re.I)
    pref = re.search(r"prefer (\w+)", text, re.I)
    parts = []
    if name:
        parts.append(f"name: {name.group(1)}")
    if pref:
        parts.append(f"prefers {pref.group(1)}")
    return "; ".join(parts) if parts else text


def route_and_store(text):
    store, reason = write_router(text)
    if store == "semantic":
        CORE["user_profile"] = _normalize_profile(text)
    elif store == "working":
        CORE["task_state"] = text
    elif store == "episodic":
        EPISODIC.append(text)
    elif store == "procedural":
        PROCEDURAL.append(text)
    return store, reason


print("write_router, _normalize_profile, route_and_store ready.")

## Step 3 - Memory as an ACTION: `memory_append` and `memory_replace` as TOOLS

Through Part 5 every tool only READ. Here memory becomes something the agent WRITES, by giving it two tools that edit the labeled CORE blocks in place. `memory_append` adds a fact to a block; `memory_replace` swaps a substring inside a block (the move that corrects a stale fact). Both refuse an unknown block and return the error as an observation the loop can read, rather than crashing, which is the error-as-observation pattern from Part 2.

These are declared with the **exact Part 1 contract**, not a new one: `TOOL_SCHEMAS` maps each tool name to a description, typed required parameters, and its function, and `validate_call` is Part 1's validator, checking the tool exists, every required arg is present, and each arg has the right type before the tool fires. `call_memory_tool` wraps validate-then-invoke so a bad call never reaches the function. The agent rewriting its own core memory (MemGPT/Letta by hand) is just the Part 1 tool loop pointed at the memory store. A read-only agent could never do this.

In [ ]:
def memory_append(block, text):
    if block not in CORE:
        return f"[error] unknown memory block '{block}'"          # error-as-observation (Part 2)
    CORE[block] = text if CORE[block] == "(empty)" else CORE[block] + "; " + text
    return f"appended to {block}"


def memory_replace(block, old, new):
    if block not in CORE:
        return f"[error] unknown memory block '{block}'"
    if old not in CORE[block]:
        return f"[error] '{old}' not found in {block}"
    CORE[block] = CORE[block].replace(old, new)
    return f"replaced '{old}' with '{new}' in {block}"


# Declared with the SAME contract as Part 1: name -> {description, typed params, fn}.
TOOL_SCHEMAS = {
    "memory_append": {
        "description": "append a fact to a labeled core-memory block",
        "parameters": {"block": {"type": "string", "required": True},
                       "text": {"type": "string", "required": True}},
        "fn": memory_append,
    },
    "memory_replace": {
        "description": "replace text inside a labeled core-memory block",
        "parameters": {"block": {"type": "string", "required": True},
                       "old": {"type": "string", "required": True},
                       "new": {"type": "string", "required": True}},
        "fn": memory_replace,
    },
}

_PY_TYPE = {"string": str, "number": (int, float), "boolean": bool}


def validate_call(name, args):                                    # the Part 1 validator
    if name not in TOOL_SCHEMAS:
        return False, f"unknown tool '{name}'"
    schema = TOOL_SCHEMAS[name]["parameters"]
    for arg, spec in schema.items():
        if spec.get("required") and arg not in args:
            return False, f"{name} is missing required arg '{arg}'"
    for arg, value in args.items():
        if arg not in schema:
            return False, f"{name} got unexpected arg '{arg}'"
        if not isinstance(value, _PY_TYPE[schema[arg]["type"]]):
            return False, f"{name} arg '{arg}' must be {schema[arg]['type']}"
    return True, None


def call_memory_tool(name, args):
    ok, err = validate_call(name, args)
    if not ok:
        return f"[invalid call] {err}"
    return TOOL_SCHEMAS[name]["fn"](**args)


print("memory_append, memory_replace, TOOL_SCHEMAS, validate_call, call_memory_tool ready.")

## Step 4 - `generate()`: the real LLM path (reference shape only)

Same device as Parts 1-5. In production the router and the controller are an LLM: you hand it an incoming item to classify, or the goal plus the memory state to decide an edit or compose a reply, and it returns the decision. `generate()` is the single swappable call that lights up the real path; the offline demo never calls it. The deterministic rule router/controller is the source of truth for everything you see in the output. SDK names and model IDs move fast, so only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM to route, decide a memory edit, or compose.
    Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[router] OPENAI_API_KEY set; the real LLM would classify/decide via generate(). "
          "Falling through to the deterministic rules so output is reproducible.")
else:
    print("[router] no OPENAI_API_KEY; using deterministic rule router/controller (offline default)")

## Demo - route four items, edit core memory as a tool, read the OS tiers, compose one reply

Everything below runs fully offline. The demo has four sections, one per mechanism. First, the **write router** sorts four incoming items into the four typed stores. Second, **memory as a tool**: the agent rewrites its own core memory with validated `memory_replace` / `memory_append` calls, including one that hits an unknown block and returns an error-as-observation. Third, the **OS-style hierarchy** reads core, recall, and archival (the last via the black-box `vector_search`). Fourth, all four memories are read at once to **compose one grounded reply**.

In [ ]:
bar = "=" * 72
print(bar)
print("THE FOUR MEMORIES  -  typed stores the agent edits itself")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[router] OPENAI_API_KEY set; the real LLM would classify/decide via generate(). "
          "Falling through to the deterministic rules so output is reproducible.")
else:
    print("[router] no OPENAI_API_KEY; using deterministic rule router/controller (offline default)")

### The taxonomy and the write router on four items

Four items arrive, one of each kind. Dana introducing herself is a durable fact (SEMANTIC -> `user_profile`, normalized to `name: Dana; prefers email`). The return request is something that happened (EPISODIC). The restocking-fee phrasing is a reusable how-to rule (PROCEDURAL; this is exactly the kind of promoted reflection Part 5 ended with). And quoting the refund now is the current focus (WORKING -> `task_state`). The router separates them so each lands where its read path expects it, which a flat buffer cannot do.

In [ ]:
print("\n" + "-" * 72)
print("THE FOUR MEMORIES (by KIND of knowledge), and the write router.")
print("-" * 72)
print("  working    what we are doing now   (volatile scratchpad)")
print("  semantic   what is TRUE            (durable facts: the user, the world)")
print("  episodic   what HAPPENED           (an event log; formalizes Part 5's buffer)")
print("  procedural how to DO a task        (learned rules; e.g. Part 5's promoted reflection)")
print()
inbox = [
    "Hi, I'm Dana and I prefer email contact.",
    "User opened a return request for ORD-3300 earlier today.",
    "Returns after the 30-day window incur a 10% restocking fee; multiply the total by 0.9.",
    "Quoting the refund for ORD-3300 now.",
]
for item in inbox:
    store, reason = route_and_store(item)
    print(f"  route -> {store:<10} ({reason})")
    print(f"           {item!r}")

print("\n  Stores after routing:")
print(f"    CORE.user_profile : {CORE['user_profile']!r}")
print(f"    CORE.task_state   : {CORE['task_state']!r}")
print(f"    EPISODIC          : {EPISODIC}")
print(f"    PROCEDURAL        : {PROCEDURAL}")

### Memory as a tool: the agent rewrites its own core memory in-loop

Now the agent edits what the router wrote. Three validated calls: `memory_replace` corrects Dana's contact preference from `prefers email` to `prefers phone` (a durable fact, fixed with one call); `memory_append` adds the order detail to `task_state`; and a third call targets a block named `preferences` that does not exist, so it returns `[error] unknown memory block 'preferences'` as an observation rather than crashing (error-as-observation, Part 2). Each call passes through Part 1's `validate_call` before it fires. The first call is the headline: a read-only agent could never correct a fact about its user, and here it does so with a single tool call.

In [ ]:
print("\n" + bar)
print("MEMORY AS A TOOL: the agent rewrites its own core memory in-loop (MemGPT/Letta).")
print("Tools use the Part 1 contract: validated before they fire.")
print(bar)
edits = [
    ("memory_replace", {"block": "user_profile", "old": "prefers email", "new": "prefers phone"}),
    ("memory_append", {"block": "task_state", "text": "order ORD-3300, $200, returned after the window"}),
    ("memory_replace", {"block": "preferences", "old": "x", "new": "y"}),     # unknown block -> error-as-observation
]
for name, args in edits:
    result = call_memory_tool(name, args)
    argstr = ", ".join(f"{k}={v!r}" for k, v in args.items())
    print(f"  {name}({argstr})")
    print(f"    -> {result}")
print(f"\n  CORE.user_profile : {CORE['user_profile']!r}")
print(f"  CORE.task_state   : {CORE['task_state']!r}")
print("  The agent corrected a durable fact about the user with a single tool call.")

### The memory hierarchy by ACCESS PATTERN: core / recall / archival

The taxonomy above sorts by KIND; this tier sketch sorts by ACCESS PATTERN, and the two are orthogonal. CORE is what stays in context: the `user_profile` and `task_state` blocks just edited. RECALL is recent and paged in on demand: the latest episodic event. ARCHIVAL is large and never held whole; it is searched. Here the archival read is the black-box `vector_search` from Step 1, the reused RAG retriever, returning the policy chunk and a similarity score. We use the chunk and the score; we do not look inside the box.

In [ ]:
print("\n" + bar)
print("THE MEMORY HIERARCHY (by ACCESS PATTERN), an OS-style sketch.")
print(bar)
print("  core      always in context  -> user_profile, task_state (above)")
print(f"  recall    recent, paged in   -> last episodic event: {EPISODIC[-1]!r}")
chunk, score = vector_search("restocking fee returns after the window")
print("  archival  large, searched    -> vector_search (black-box RAG; not re-derived here):")
print(f"             query 'restocking fee returns after the window'")
print(f"             -> {chunk!r} (score={score})")

### Composing one grounded reply from all four memories

The payoff: a single reply that draws on every store at once. SEMANTIC supplies the name and contact preference (Dana, prefers phone). WORKING supplies the task and order detail. PROCEDURAL supplies the rule that turns `$200` into `$180` (multiply by 0.9, the Part 5 lesson now living in its own store). ARCHIVAL supplies the policy chunk that grounds the basis. EPISODIC supplies the event the reply is responding to. Each fact comes from the store that holds its kind, which is exactly what a flat buffer could not give us.

In [ ]:
print("\n" + bar)
print("COMPOSING A REPLY from all four memories at once.")
print(bar)
refund = round(200.0 * 0.9, 2)                          # procedural rule applied
print(f"  semantic   (user_profile): {CORE['user_profile']}")
print(f"  working    (task_state)  : {CORE['task_state']}")
print(f"  procedural (rule)        : {PROCEDURAL[0]}")
print(f"  archival   (RAG chunk)   : {chunk}")
print(f"  episodic   (event)       : {EPISODIC[-1]}")
print(f"\n  REPLY: Hi Dana, for ORD-3300 returned after the 30-day window your refund is")
print(f"         ${refund:.2f} (the 10% restocking fee applies). We will call you by phone.")

print("\n" + bar)
print("Done. A flat buffer cannot tell what happened from what is true from how to")
print("act. Four typed stores + a write router separate them; memory_append and")
print("memory_replace make memory an ACTION the agent takes, not just state it reads;")
print("and archival is one black-box retrieval call, not a vector DB we rebuilt.")
print(bar)

## Wrap-up

Part 5's reflection buffer was a flat list: enough to carry one lesson forward, but unable to separate what HAPPENED from what is TRUE from how to DO a task, and the agent could only ever READ its tools. This part fixed both.

- **Four typed memories** sort knowledge by KIND: WORKING (current focus), SEMANTIC (durable facts), EPISODIC (the event log, which formalizes Part 5's buffer), and PROCEDURAL (learned rules, which is where Part 5's promoted reflection now lives). A **write router** classifies each item and sends it to the right store.
- **Memory as an action**: `memory_append` and `memory_replace`, declared with the Part 1 tool contract and validated before firing, let the agent rewrite its own labeled CORE blocks in-loop (MemGPT/Letta by hand). The agent is no longer a read-only consumer of state.
- An orthogonal **OS-style hierarchy** placed memory by ACCESS PATTERN: CORE (always in context), RECALL (recent, paged in), and ARCHIVAL (large, searched). The archival read REUSES RAG retrieval as a black-box `vector_search`; we did not re-derive embeddings, similarity, or a vector DB.

The reply at the end read all four stores at once to ground a single answer, which a flat buffer could not do.

**Part 7 - Compaction and forgetting** is next. These stores grow without bound: every event appends to EPISODIC, every edit lengthens a CORE block, and context is finite. Part 7 adds the other half of a memory system, deciding what to summarize, what to demote from core to archival, and what to drop, so memory stays useful instead of just large.